In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

!pip install tensorflow==2.12.0
!pip install keras

In [3]:
!pip install pandas
!pip install opencv-python

In [4]:
!pip install pandas pillow

In [5]:
import numpy as np
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import cv2
import os
from tensorflow import keras
import glob as gb
import pandas as pd
from tensorflow.keras.preprocessing import image_dataset_from_directory
# Import RandomFlip and RandomRotation from the preprocessing module
from tensorflow.keras.layers.experimental.preprocessing import RandomFlip, RandomRotation
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.layers import Conv2D, MaxPool2D, LeakyReLU



IMG_SIZE = (299, 299) # resolution
directory = "/content/drive/MyDrive/Original Dataset"
BATCH_SIZE = 128
train_dataset = image_dataset_from_directory(directory,
                                             shuffle=True,
                                             labels='inferred',
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.1,
                                             subset='training',
                                             color_mode='rgb',
                                             seed=42)
validation_dataset = image_dataset_from_directory(directory,
                                             shuffle=True,
                                             labels='inferred',
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.1,
                                             subset='validation',
                                             color_mode='rgb',
                                             seed=42)



class_names = train_dataset.class_names
class_names


sns.set_theme(style="whitegrid")
Data_imbalance = []
for folder in os.listdir(directory):
    files = gb.glob(pathname=str(directory + "/" + folder +"/*.*"))
    Data_imbalance.append(len(files))
plt.figure(figsize=(13,7))
sns.barplot(x=['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy'], y=Data_imbalance, palette="rocket")
plt.show()



total = 0
for i in range(0,len(Data_imbalance)) :
    total +=Data_imbalance[i]

weight_for_0 = (1 / Data_imbalance[0]) * (total / 4.0)
weight_for_1 = (1 / Data_imbalance[1]) * (total / 4.0)

class_weight = {0: weight_for_0, 1: weight_for_1}

print('Weight for class 0: {:.2f}'.format(weight_for_0))
print('Weight for class 1: {:.2f}'.format(weight_for_1))


data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal"),
  tf.keras.layers.RandomRotation(0.2),
  tf.keras.layers.RandomZoom(0.2),
  tf.keras.layers.RandomHeight(0.2),
  tf.keras.layers.RandomWidth(0.2),
])


for image, _ in train_dataset.take(1):
    plt.figure(figsize=(10, 10))
    first_image = image[0]
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        augmented_image = data_augmentation(tf.expand_dims(first_image, 0))
        plt.imshow(augmented_image[0] / 255)
        plt.axis('off')



AUTOTUNE = tf.data.experimental.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)


NotFoundError: Could not find directory /content/drive/MyDrive/Original Dataset

In [ ]:
from tensorflow.keras import layers, models

def C3K2(x, filters, shortcut=True, n=3):
    for _ in range(n):
        x_short = x
        x = layers.Conv2D(filters, (1, 1), padding='same')(x)
        x = LeakyReLU()(x)
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = LeakyReLU()(x)
        if shortcut:
            x = layers.Add()([x, x_short])
    return x

def YOLOv11(input_shape=(299, 299, 3), num_classes=2):
    inputs = layers.Input(shape=input_shape)

    # Backbone
    x = layers.Conv2D(32, 3, strides=1, padding='same')(inputs)
    x = LeakyReLU()(x)
    x = layers.Conv2D(64, 3, strides=2, padding='same')(x)
    x = LeakyReLU()(x)
    x = layers.Conv2D(128, 3, strides=2, padding='same')(x)
    x = LeakyReLU()(x)
    x = C3K2(x, 128, shortcut=False)
    skip1 = x
    x = layers.Conv2D(256, 3, strides=2, padding='same')(x)
    x = LeakyReLU()(x)
    x = C3K2(x, 256)
    skip2 = x
    x = layers.Conv2D(512, 3, strides=2, padding='same')(x)
    x = LeakyReLU()(x)
    x = C3K2(x, 512)

    # Neck (simplified)
    x = layers.UpSampling2D()(x)
    x = layers.Concatenate()([x, skip2])
    x = C3K2(x, 256, shortcut=False)

    x = layers.UpSampling2D()(x)
    x = layers.Concatenate()([x, skip1])
    x = C3K2(x, 128, shortcut=False)

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs)

model = YOLOv11()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=25,
    class_weight=class_weight
)


In [ ]:
# Plot accuracy and loss
plt.figure(figsize=(14, 6))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', marker='o')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='o')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Get true labels and predictions from validation dataset
y_true = []
y_pred = []

for images, labels in validation_dataset:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Class names
print("Class names:", class_names)

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()
